# 의료사고로 고소된 의사는 실제로 어디까지 갈까? (재현 노트북)

DataPopcorn 테크블로그 글 [datapopcorn.ai/blog/medical-accident-criminal-data](https://datapopcorn.ai/blog/medical-accident-criminal-data)의 재현 노트북입니다. `런타임 → 모두 실행`을 누르면 글에 나온 숫자와 차트 5개가 다시 만들어집니다.

- 데이터: 공공 보고서의 표를 CSV로 옮긴 것(각 행에 출처 표기)과 유튜브 검색 반응 요약(제목 제외, 주제 분류만)입니다. 2026-10-25 정리.
- 서로 다른 통계(조정 신청·피의자·판결)를 나란히 놓을 뿐, 단계 간 비율로 나누지 않습니다.
- 이 노트북은 특정 사건의 과실·책임을 판단하지 않습니다. API 키 없이 실행됩니다. 보고서 원문 PDF는 README의 주소에서 받으세요.

In [ ]:
# 0. 준비: 한글 폰트 + 데이터 내려받기 (로컬에서는 DATA_DIR 환경변수로 폴더 지정)
import os, sys, urllib.request, urllib.parse
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.system('apt-get -qq install -y fonts-nanum > /dev/null')
for f in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf', '/System/Library/Fonts/AppleSDGothicNeo.ttc', 'C:/Windows/Fonts/malgun.ttf']:
    if os.path.exists(f):
        os.environ['KFONT'] = f; break
FILES = ['suspects_doctor_by_year.csv', 'judgments_by_year.csv', 'verdicts_192.csv', 'departments_192.csv', 'kmedi_requests.csv', 'youtube_reaction.json']
BASE = os.environ.get('DATA_DIR', 'https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/medical-accident-criminal-data/data')
if BASE.startswith('http'):
    os.makedirs('data', exist_ok=True)
    for fn in FILES:
        urllib.request.urlretrieve(f'{BASE}/{urllib.parse.quote(fn)}', f'data/{fn}')
    DATA = 'data'
else:
    DATA = BASE
os.environ['DATA'] = DATA; os.environ['FIG'] = 'fig'; os.environ['OUT'] = 'summary.json'
print('데이터 폴더:', DATA, sorted(os.listdir(DATA)))

In [ ]:
import json,os,pandas as pd,numpy as np,matplotlib
matplotlib.use('Agg');import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
FONT=os.environ.get('KFONT');fm.fontManager.addfont(FONT)
plt.rcParams.update({'font.family':fm.FontProperties(fname=FONT).get_name(),'axes.unicode_minus':False,'figure.dpi':150,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
C={'main':'#E8590C','sub':'#1C7ED6','gray':'#868E96','light':'#CED4DA'}
D=os.environ.get('DATA','data');FIG=os.environ.get('FIG','fig');os.makedirs(FIG,exist_ok=True);S={}
sus=pd.read_csv(f'{D}/suspects_doctor_by_year.csv');jud=pd.read_csv(f'{D}/judgments_by_year.csv');ver=pd.read_csv(f'{D}/verdicts_192.csv');dep=pd.read_csv(f'{D}/departments_192.csv');km=pd.read_csv(f'{D}/kmedi_requests.csv')
yt=pd.DataFrame(json.load(open(f'{D}/youtube_reaction.json'))['rows'])

## 데이터 읽기와 요약값 계산
연평균은 2019~2023년 5년 기준입니다. 실형은 금고·징역(집행유예 제외)입니다.

In [ ]:
# ---------- summary ----------
S['kmedi_avg_2019_2023']=round(km[km.year<=2023].requests.mean(),1);S['kmedi_2025']=int(km[km.year==2025].requests.iloc[0])
S['suspects_avg_2019_2023']=round(sus.suspects.mean(),1);S['suspects_by_year']=dict(zip(sus.year.astype(str),sus.suspects))
S['defendants_total']=int(ver.people.sum());S['defendants_avg']=round(ver.people.sum()/5,1);S['cases_total']=int(jud.cases.sum());S['cases_avg']=round(jud.cases.mean(),1)
real=int(ver[ver.group=='실형'].people.sum());S['prison_total']=real;S['prison_avg']=round(real/5,1)
S['verdict_pct']={r:round(p/192*100,1) for r,p in zip(ver.result,ver.people)}
S['acquittal_pct']=S['verdict_pct']['무죄'];S['fine_pct']=S['verdict_pct']['벌금']
S['dept']={d:int(p) for d,p in zip(dep.dept,dep.people)};S['dept_pct']={d:round(p/192*100,1) for d,p in zip(dep.dept,dep.people)}
th=yt.groupby('theme').views.agg(['count','sum']).sort_values('sum',ascending=False);S['yt_total']=int(yt.views.sum());S['yt_theme']={k:{'n':int(r['count']),'views':int(r['sum']),'pct':round(r['sum']/yt.views.sum()*100,1)} for k,r in th.iterrows()}

## ① 단계별 규모 (Step 3)
조정 신청(건), 직업 '의사' 업무상과실치사상 피의자(명), 1심 판결 피고인(명), 실형(명)을 나란히 놓습니다. 집계 범위가 달라 전환율이 아닙니다.

In [ ]:
# ---------- 1. funnel ----------
st=[('의료분쟁 조정 신청',S['kmedi_avg_2019_2023'],'건'),("업무상과실치사상 피의자\n(직업 '의사')",S['suspects_avg_2019_2023'],'명'),('형사재판 1심 판결\n(의사·치과·한의 피고인)',S['defendants_avg'],'명'),('금고·징역 실형',S['prison_avg'],'명')]
fig,ax=plt.subplots(figsize=(11,4.6));y=np.arange(len(st))[::-1]
ax.barh(y,[s[1] for s in st],color=[C['light'],C['sub'],C['main'],C['main']])
for yy,s in zip(y,st):ax.text(s[1]+30,yy,f"{s[1]:,.1f}{s[2]}".replace('.0',''),va='center',fontsize=11)
ax.set_yticks(y);ax.set_yticklabels([s[0] for s in st]);ax.set_xlim(0,2700);ax.set_xlabel('2019~2023 연평균')
ax.set_title(f"① 수사 대상은 한 해 {S['suspects_avg_2019_2023']:.0f}명, 재판 판결은 {S['defendants_avg']}명, 실형은 {S['prison_avg']}명\n의료사고 사법 단계별 규모 (서로 다른 통계를 나란히 놓은 것, 전환율 아님)",loc='left')
fig.text(0.01,-0.05,'자료: 한국의료분쟁조정중재원 통계연보, 대검찰청 범죄분석(보사연 보고서 표3-1-1 재인용), 보사연·한국형사법무정책연구원 판결문 분석(표3-2-11·12). 판결 분석은 약식명령·불기소 제외',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig1_funnel.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig1_funnel.png'))

## ② 연도별 피의자와 1심 판결 (Step 4)
최근 연도 판결은 재판 진행 중 사건 때문에 적게 나올 수 있습니다.

In [ ]:
# ---------- 2. by year ----------
fig,ax=plt.subplots(1,2,figsize=(12,3.9))
ax[0].bar(sus.year.astype(str),sus.suspects,color=C['sub']);[ax[0].text(i,v+10,f'{v}',ha='center') for i,v in enumerate(sus.suspects)];ax[0].set_title("직업 '의사' 업무상과실치사상 피의자(명)",loc='left',fontsize=11);ax[0].set_ylim(0,1000)
ax[1].bar(jud.year.astype(str),jud.cases,color=C['main']);[ax[1].text(i,v+1,f'{v}',ha='center') for i,v in enumerate(jud.cases)];ax[1].set_title('의료인 업무상과실치사상 1심 판결(건)',loc='left',fontsize=11);ax[1].set_ylim(0,60)
fig.suptitle('② 연도별로 봐도 수사 대상은 수백 명, 1심 판결은 수십 건',x=0.01,ha='left',fontsize=12.5)
fig.text(0.01,-0.05,'자료: 대검찰청 범죄분석(보사연 표3-1-1), 보사연 판결문 분석(표3-2-3). 축 단위가 다름',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig2_by_year.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig2_by_year.png'))

## ③ 1심 선고 결과 192명 (Step 5)
보사연·한국형사법무정책연구원 판결문 분석 표3-2-11.

In [ ]:
# ---------- 3. verdicts ----------
v2=ver.sort_values('people');fig,ax=plt.subplots(figsize=(10,4.2))
ax.barh(v2.result,v2.people,color=[C['main'] if g=='실형' else (C['sub'] if r=='무죄' else C['light']) for r,g in zip(v2.result,v2.group)])
for i,p in enumerate(v2.people):ax.text(p+0.8,i,f'{p}명 ({p/192*100:.1f}%)',va='center',fontsize=9.5)
ax.set_xlim(0,85);ax.set_title(f"③ 1심 판결 192명 중 벌금 {S['fine_pct']}%, 무죄 {S['acquittal_pct']}%, 실형(금고·징역)은 16명",loc='left')
fig.text(0.01,-0.05,'자료: 보사연·한국형사법무정책연구원 판결문 분석 표3-2-11(2019~2023 1심). 벌금 최빈 500만 원(표3-2-13)',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig3_verdicts.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig3_verdicts.png'))

## ④ 진료과 분포 (Step 6)
표3-2-37. 진료과별 의사 수 대비가 아닌 단순 인원입니다.

In [ ]:
# ---------- 4. departments ----------
d2=dep.iloc[::-1];fig,ax=plt.subplots(figsize=(10,4.6))
ax.barh(d2.dept,d2.people,color=[C['main'] if d in ('소아청소년과','응급의학과') else C['light'] for d in d2.dept])
for i,p in enumerate(d2.people):ax.text(p+0.4,i,f'{p}명',va='center',fontsize=9)
ax.set_title('④ 재판에 선 192명 중 정형외과·성형외과가 가장 많고, 소아청소년과 7명·응급의학과 9명',loc='left')
fig.text(0.01,-0.05,'자료: 보사연 판결문 분석 표3-2-37(진료항목). 진료과별 의사 수 대비 비율이 아니라 단순 인원',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig4_departments.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig4_departments.png'))

## ⑤ 유튜브 반응 주제 분류 (Step 2)
'아주대병원 압수수색' 검색 상위 영상의 조회수를 제목 단어로 나눈 결과입니다(제목은 싣지 않음).

In [ ]:
# ---------- 5. reaction ----------
th2=th.reset_index();fig,ax=plt.subplots(figsize=(10,3.6))
ax.barh(th2.theme[::-1],th2['sum'][::-1]/1e4,color=C['light'])
for i,(n,s) in enumerate(zip(th2['count'][::-1],th2['sum'][::-1])):ax.text(s/1e4+10,i,f'{s/1e4:.0f}만 회 · {n}개',va='center',fontsize=9.5)
ax.set_xlabel('조회수(만 회)');ax.set_xlim(0,1100);ax.set_title(f"⑤ 사람들이 본 영상: '아주대병원 압수수색' 검색 상위 {len(yt)}개, 합 {S['yt_total']/1e4:.0f}만 회",loc='left')
fig.text(0.01,-0.06,'자료: 유튜브 검색 결과(2026-10-25 조회). 제목 단어로 주제를 나눔(한 영상이 여러 주제면 앞 주제로). 제목은 싣지 않음',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig5_reaction.png',bbox_inches='tight');plt.close()

from IPython.display import Image, display
display(Image(f'{FIG}/fig5_reaction.png'))

In [ ]:
import json
print(json.dumps(S, ensure_ascii=False, indent=1))